In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import math
import random
from tqdm import tqdm
from scipy.ndimage import uniform_filter
from skimage.metrics import structural_similarity as ssim

CONFIG = {
    "VOLUME_PATH": "sliepner.npy",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "BATCH_SIZE": 4,
    "IMG_SIZE": 128,      
    "UNET_DIM": 64,
    "SEED": 42,
    "DIFFUSION_STEPS": 1000, 
    "EXTRACTION_TIMESTEP": 100, 
    "TRAIN_EPOCHS": 20,
    "FINE_TUNE_EPOCHS": 2,
    "LR": 1e-4,
    "TIME_LIMIT": 600,
    "N_EXAMPLES": 3,
    
    # Attributes to Run
    "RUN_ATTRIBUTES": ["rms", "semblance", "dip", "curvature"],
    "RUN_ATTRIBUTES": ["rms", "semblance"],
    
    # Attribute Parameters
    "RMS_PARAMS": {"window": 7},
    "SEMBLANCE_PARAMS": {"window_z": 5, "window_x": 3},
    "DIP_PARAMS": {}, # No specific params for current simple implementation
    "CURVATURE_PARAMS": {} # No specific params for current simple implementation
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])

print(f"--- Configuration Loaded. Device: {CONFIG['DEVICE']} ---")

def calculate_rms(volume, window=9):
    print(f"   -> Calculating RMS Attribute (Window={window})...")
    squared = volume ** 2
    # Apply window along the time axis (dimension 2)
    # Using (3, 3) for spatial smoothing and 'window' for time smoothing
    mse = uniform_filter(squared, size=(3, 3, window))
    rms = np.sqrt(mse)
    
    p99 = np.percentile(rms, 99)
    if p99 > 0:
        rms = np.clip(rms / p99, 0, 1) * 2 - 1 
    return rms.astype(np.float32)

def calculate_semblance(volume, window_z=5, window_x=3):
    print(f"   -> Calculating Semblance (Window Z={window_z}, X={window_x})...")
    
    # 1. Local Mean (window_x spatial, window_z time)
    local_mean = uniform_filter(volume, size=(window_x, window_x, window_z))
    
    # 2. Local Mean Square
    local_mean_sq = uniform_filter(volume**2, size=(window_x, window_x, window_z))
    
    # 3. Variance
    local_var = local_mean_sq - local_mean**2
    
    # Semblance proxy
    semb = 1.0 / (1.0 + local_var * 100)
    
    # Normalize
    s_min, s_max = np.min(semb), np.max(semb)
    if s_max > s_min:
        semb = (semb - s_min) / (s_max - s_min)
        semb = semb * 2 - 1
    
    return semb.astype(np.float32)

def calculate_dip(volume):
    print("   -> Calculating Dip...")
    gx, gy, gz = np.gradient(volume)
    
    spatial_grad_mag = np.sqrt(gx**2 + gy**2)
    dip_angle = np.arctan(spatial_grad_mag / (np.abs(gz) + 1e-8)) 
    dip_degrees = np.degrees(dip_angle)
    
    p99 = np.percentile(dip_degrees, 99)
    if p99 > 0:
        dip_norm = np.clip(dip_degrees / p99, 0, 1) * 2 - 1
    else:
        dip_norm = np.zeros_like(dip_degrees)
        
    return dip_norm.astype(np.float32)

def calculate_curvature(volume):
    print("   -> Calculating Curvature (Mean)...")
    gx, gy, _ = np.gradient(volume)
    
    gxx = np.gradient(gx, axis=0)
    gyy = np.gradient(gy, axis=1)
    
    curvature = gxx + gyy
    
    robust_max = np.percentile(np.abs(curvature), 99)
    if robust_max > 0:
        curvature = np.clip(curvature / robust_max, -1, 1)
        
    return curvature.astype(np.float32)

class SeismicAttributeDataset(Dataset):
    def __init__(self, mode='amplitude'):
        self.mode = mode
        self.img_size = CONFIG["IMG_SIZE"]
        
        if not hasattr(self, 'vol_amp'):
            vol_path = CONFIG["VOLUME_PATH"]
            
            if os.path.exists(vol_path):
                print(f"\n--- Loading Seismic Data from {vol_path} ---")
                raw_vol = np.load(vol_path).astype(np.float32)
                
                limit = CONFIG["TIME_LIMIT"]
                if len(raw_vol.shape) > 2 and raw_vol.shape[2] > limit:
                    print(f"   -> Cropping time axis to first {limit} samples...")
                    raw_vol = raw_vol[:, :, :limit]
                
                robust_max = np.percentile(np.abs(raw_vol), 99)
                if robust_max == 0: robust_max = 1.0
                raw_vol = np.clip(raw_vol / robust_max, -1, 1)
                
                self.vol_amp = raw_vol
                CONFIG["IMG_H"] = self.vol_amp.shape[1]
                CONFIG["IMG_W"] = self.vol_amp.shape[2]

            else:
                print(f"\n--- Warning: {vol_path} not found. Generating Synthetic Data. ---")
                limit = CONFIG["TIME_LIMIT"]
                self.vol_amp = np.random.randn(limit, self.img_size, self.img_size).astype(np.float32)
                CONFIG["IMG_H"] = self.img_size
                CONFIG["IMG_W"] = self.img_size

            # Compute Attributes based on Config
            # We pre-compute all to allow easy switching, or could lazy load
            self.vol_rms = calculate_rms(self.vol_amp, **CONFIG["RMS_PARAMS"])
            self.vol_semb = calculate_semblance(self.vol_amp, **CONFIG["SEMBLANCE_PARAMS"])
            self.vol_dip = calculate_dip(self.vol_amp)
            self.vol_curvature = calculate_curvature(self.vol_amp)
            
            print("--- Data Preparation Complete ---\n")

    def set_mode(self, mode):
        self.mode = mode
        print(f"Dataset switched to mode: {mode.upper()}")

    def __len__(self):
        return self.vol_amp.shape[0]

    def __getitem__(self, idx):
        if self.mode == 'amplitude':
            img = self.vol_amp[idx]
        elif self.mode == 'rms':
            img = self.vol_rms[idx]
        elif self.mode == 'semblance':
            img = self.vol_semb[idx]
        elif self.mode == 'dip':
            img = self.vol_dip[idx]
        elif self.mode == 'curvature':
            img = self.vol_curvature[idx]
            
        img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0) 
        return img_t

class DiffusionUtils:
    def __init__(self, steps=1000, device="cuda"):
        self.steps = steps
        self.device = device
        self.betas = torch.linspace(0.0001, 0.02, steps).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)

    def q_sample(self, x_0, t, noise=None):
        if noise is None: noise = torch.randn_like(x_0)
        return (self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) * x_0 + 
                self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise), noise

    def predict_start_from_noise(self, x_t, t, noise):
        return (x_t - self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise) / self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=8):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding=1)
        self.norm = nn.GroupNorm(groups, dim_out)
        self.act = nn.SiLU()
    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if scale_shift is not None:
            scale, shift = scale_shift
            x = x * (scale + 1) + shift
        return self.act(x)

class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, *, time_emb_dim=None):
        super().__init__()
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, dim_out * 2)) if time_emb_dim else None
        self.block1 = Block(dim, dim_out)
        self.block2 = Block(dim_out, dim_out)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()
    def forward(self, x, time_emb=None):
        scale_shift = None
        if self.mlp is not None and time_emb is not None:
            time_emb = self.mlp(time_emb)
            time_emb = time_emb.unsqueeze(-1).unsqueeze(-1)
            scale_shift = time_emb.chunk(2, dim=1)
        h = self.block1(x, scale_shift=scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class UNet(nn.Module):
    def __init__(self, dim=CONFIG["UNET_DIM"], out_dim=1, channels=1): 
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(dim), nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim * 4),
        )
        self.init_conv = nn.Conv2d(channels, dim, 7, padding=3)
        self.downs = nn.ModuleList([
            ResnetBlock(dim, dim, time_emb_dim=dim*4),
            ResnetBlock(dim, dim*2, time_emb_dim=dim*4),
        ])
        self.down_samples = nn.ModuleList([nn.Conv2d(dim, dim, 4, 2, 1), nn.Conv2d(dim*2, dim*2, 4, 2, 1)])
        
        self.mid_block1 = ResnetBlock(dim*2, dim*2, time_emb_dim=dim*4)
        self.mid_block2 = ResnetBlock(dim*2, dim*2, time_emb_dim=dim*4)
        
        self.ups = nn.ModuleList([
            ResnetBlock(dim*4, dim*2, time_emb_dim=dim*4),
            ResnetBlock(dim*3, dim, time_emb_dim=dim*4), 
        ])
        self.up_samples = nn.ModuleList([
            nn.ConvTranspose2d(dim*2, dim*2, 4, 2, 1), 
            nn.ConvTranspose2d(dim*2, dim*2, 4, 2, 1)
        ])
        self.final_conv = nn.Conv2d(dim, out_dim, 1)

    def forward(self, x, time):
        img_h, img_w = x.shape[-2:]
        t = self.time_mlp(time)
        x = self.init_conv(x)
        h_vals = []
        
        for block, sample in zip(self.downs, self.down_samples):
            x = block(x, t)
            h_vals.append(x)
            x = sample(x)
            
        x = self.mid_block1(x, t)
        x = self.mid_block2(x, t)
            
        for block, sample in zip(self.ups, self.up_samples):
            x = sample(x)
            h = h_vals.pop()
            if x.shape[2:] != h.shape[2:]: 
                x = F.interpolate(x, size=h.shape[2:], mode='nearest')
            x = torch.cat((x, h), dim=1)
            x = block(x, t)
            
        out = self.final_conv(x)
        if out.shape[-2:] != (img_h, img_w):
            out = F.interpolate(out, size=(img_h, img_w), mode='bilinear', align_corners=False)
        return out

def calculate_tv(img):
    tv_h = np.sum(np.abs(img[1:, :] - img[:-1, :]))
    tv_w = np.sum(np.abs(img[:, 1:] - img[:, :-1]))
    return (tv_h + tv_w) / img.size

def compute_metrics(real, recon):
    mse = np.mean((real - recon) ** 2)
    if mse == 0:
        psnr = 100.0
    else:
        psnr = 20 * math.log10(2.0 / math.sqrt(mse))
    
    score_ssim = ssim(real, recon, data_range=2.0)
    tv_in = calculate_tv(real)
    tv_out = calculate_tv(recon)
    
    return psnr, score_ssim, tv_in, tv_out

def get_quality_assessment(psnr, ssim, tv_change):
    if ssim >= 0.8 and tv_change <= -10:
        return "Excellent"
    elif ssim >= 0.6 and tv_change < 0:
        return "Good"
    elif ssim < 0.6:
        return "Bad (Structure Loss)"
    elif tv_change > 0:
        return "Bad (Noise Added)"
    else:
        return "Average"

def apply_and_visualize(dataset, diff_model, diff_utils, stage_name="Final"):
    mode_name = dataset.mode.upper()
    print(f"\n--- Applying Model to {mode_name} ({stage_name}) ---")
    
    n_examples = CONFIG["N_EXAMPLES"]
    total_len = len(dataset)
    indices = np.linspace(50, total_len - 50, n_examples, dtype=int)
    
    fig, axs = plt.subplots(n_examples, 3, figsize=(24, 6 * n_examples))
    plt.subplots_adjust(left=0.05, right=0.92, top=0.95, bottom=0.05, wspace=0.1, hspace=0.2)
    
    diff_model.eval()
    
    print(f"\n>>> Quality Report ({stage_name}) <<<")

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_t = dataset[idx].unsqueeze(0).to(CONFIG["DEVICE"])
            
            t_val = CONFIG["EXTRACTION_TIMESTEP"]
            t = torch.tensor([t_val], device=CONFIG["DEVICE"]).long()
            noise = torch.randn_like(img_t)
            noisy_img, _ = diff_utils.q_sample(img_t, t, noise)
            
            pred_noise = diff_model(noisy_img, t)
            diff_recon = diff_utils.predict_start_from_noise(noisy_img, t, pred_noise)
            
            real_np = img_t.cpu().squeeze().numpy()
            diff_np = diff_recon.cpu().squeeze().detach().numpy()
            
            diff_map = real_np - diff_np

            psnr, score_ssim, tv_in, tv_out = compute_metrics(real_np, diff_np)
            tv_change = ((tv_out - tv_in) / tv_in) * 100
            quality = get_quality_assessment(psnr, score_ssim, tv_change)
            
            print(f"Slice {idx} | PSNR: {psnr:.2f}dB | SSIM: {score_ssim:.3f} | TV Change: {tv_change:.1f}% -> Quality: {quality}")

            im1 = axs[i, 0].imshow(real_np.T, cmap='gray', vmin=-1, vmax=1)
            axs[i, 0].set_title(f"Slice {idx}: INPUT {mode_name}\nTV (Noise): {tv_in:.4f}", 
                               fontsize=14, fontweight='bold')
            
            title_str = (f"ENHANCED\n"
                         f"SSIM: {score_ssim:.3f} | PSNR: {psnr:.2f}dB\n"
                         f"TV Change: {tv_change:.1f}%")
            
            im2 = axs[i, 1].imshow(diff_np.T, cmap='gray', vmin=-1, vmax=1)
            axs[i, 1].set_title(title_str, fontsize=14, color='darkgreen', fontweight='bold')
            
            im3 = axs[i, 2].imshow(diff_map.T, cmap='gray', vmin=-1, vmax=1)
            axs[i, 2].set_title(f"DIFFERENCE (Input - Enhanced)\nShows Removed Noise/Artifacts", 
                                fontsize=14, color='darkred', fontweight='bold')

    cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7]) 
    cbar = fig.colorbar(axs[0, 0].images[0], cax=cbar_ax)
    cbar.set_label('Attribute Value (Normalized)')

    filename = f'{dataset.mode}_uplift_{stage_name.lower().replace(" ", "_")}.png'
    print(f"Saving visualization to {filename}...")
    plt.suptitle(f"{mode_name} Attribute Uplift - {stage_name}", fontsize=20, y=0.99)
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

def train_models(dataset, diff_model, diff_utils, epochs, label="Training"):
    loader = DataLoader(dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
    opt_diff = torch.optim.Adam(diff_model.parameters(), lr=CONFIG["LR"])
    mse = nn.MSELoss()
    
    print(f"\n--- Starting {label} on {dataset.mode.upper()} data for {epochs} epochs ---")
    
    for epoch in range(epochs):
        diff_loss_acc = 0
        diff_model.train() 
        
        for imgs in loader:
            imgs = imgs.to(CONFIG["DEVICE"])
            
            t = torch.randint(0, diff_utils.steps, (imgs.shape[0],), device=CONFIG["DEVICE"]).long()
            noise = torch.randn_like(imgs)
            noisy_imgs, _ = diff_utils.q_sample(imgs, t, noise)
            
            opt_diff.zero_grad()
            pred_noise = diff_model(noisy_imgs, t)
            loss_diff = mse(pred_noise, noise)
            loss_diff.backward()
            opt_diff.step()
            diff_loss_acc += loss_diff.item()
            
        if (epoch+1) % 5 == 0:
            print(f"{label} Epoch {epoch+1}/{epochs} | Loss: {diff_loss_acc/len(loader):.5f}")
            apply_and_visualize(dataset, diff_model, diff_utils, stage_name=f"{label} Epoch {epoch+1}")

def run_experiment():
    dataset = SeismicAttributeDataset(mode='amplitude')
    diff_utils = DiffusionUtils(steps=CONFIG["DIFFUSION_STEPS"], device=CONFIG["DEVICE"])
    unet = UNet(dim=CONFIG["UNET_DIM"]).to(CONFIG["DEVICE"])
    
    # 1. Pre-training on Amplitude
    train_models(dataset, unet, diff_utils, epochs=CONFIG["TRAIN_EPOCHS"], label="Pre-training")
    
    # 2. Dynamic Attribute Loop
    for attr in CONFIG["RUN_ATTRIBUTES"]:
        dataset.set_mode(attr)
        label_text = f"Fine-tuning {attr.upper()}"
        train_models(dataset, unet, diff_utils, epochs=CONFIG["FINE_TUNE_EPOCHS"], label=label_text)
        apply_and_visualize(dataset, unet, diff_utils, stage_name=f"Final {attr.upper()} Fine-Tuned")

if __name__ == "__main__":
    run_experiment()